## Import DuckDB

In [24]:
import duckdb

## Read Data

In [25]:
DATA_PATH = "../../data/land_registry"

In [26]:
land_registry_data = duckdb.sql(
    f"""
    SELECT
        column00 AS id,
        column01 AS price,
        column02 AS date,
        column03 AS postcode,
        column04 AS property_type,
        column05 AS old_new,
        column06 AS duration,
        column07 AS paon,
        column08 AS saon,
        column09 AS street,
        column10 AS locality,
        column11 AS town_city,
        column12 AS district,
        column13 AS county,
        column14 AS ppd_category,
        column15 AS record_type
    FROM read_csv('{DATA_PATH}/pp-*.csv', header = false)
    """
)

In [27]:
land_registry_data

┌────────────────────────────────────────┬────────┬─────────────────────┬──────────┬───────────────┬─────────┬──────────┬───────────────┬─────────────┬──────────────────┬─────────────┬───────────────┬──────────────────────┬──────────────────────┬──────────────┬─────────────┐
│                   id                   │ price  │        date         │ postcode │ property_type │ old_new │ duration │     paon      │    saon     │      street      │  locality   │   town_city   │       district       │        county        │ ppd_category │ record_type │
│                varchar                 │ int64  │      timestamp      │ varchar  │    varchar    │ varchar │ varchar  │    varchar    │   varchar   │     varchar      │   varchar   │    varchar    │       varchar        │       varchar        │   varchar    │   varchar   │
├────────────────────────────────────────┼────────┼─────────────────────┼──────────┼───────────────┼─────────┼──────────┼───────────────┼─────────────┼──────────────────┼──

In [28]:
duckdb.sql("DESCRIBE FROM land_registry_data")

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ price         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ date          │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ postcode      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ property_type │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ old_new       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ duration      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ paon          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ saon          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ street        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NU

In [29]:
duckdb.sql(
    """
    SELECT property_type, COUNT(*) AS count
    FROM land_registry_data
    GROUP BY property_type
    """
)

┌───────────────┬────────┐
│ property_type │ count  │
│    varchar    │ int64  │
├───────────────┼────────┤
│ T             │ 497490 │
│ F             │ 301617 │
│ D             │ 406028 │
│ S             │ 492364 │
│ O             │  83884 │
└───────────────┴────────┘

In [30]:
land_registry_data = duckdb.sql(
    """
    SELECT * REPLACE (
        CASE property_type
            WHEN 'D' THEN 'Detached'
            WHEN 'S' THEN 'Semi-Detached'
            WHEN 'T' THEN 'Terraced'
            WHEN 'F' THEN 'Flat'
            WHEN 'O' THEN 'Other'
            ELSE property_type
        END AS property_type
    )
    FROM land_registry_data
    """
)

In [31]:
duckdb.sql(
    """
    SELECT property_type, COUNT(*) AS count
    FROM land_registry_data
    GROUP BY property_type
    """
)

┌───────────────┬────────┐
│ property_type │ count  │
│    varchar    │ int64  │
├───────────────┼────────┤
│ Other         │  83884 │
│ Detached      │ 406028 │
│ Flat          │ 301617 │
│ Semi-Detached │ 492364 │
│ Terraced      │ 497490 │
└───────────────┴────────┘

In [32]:
land_registry_data = duckdb.sql(
    """
    SELECT * FROM land_registry_data
    WHERE property_type != 'Other'
    """
)

In [33]:
land_registry_data = duckdb.sql(
    """
    SELECT *, year(date) AS year
    FROM land_registry_data
    """
)

In [34]:
duckdb.sql(
    """
    SELECT id, date, year, property_type, price
    FROM land_registry_data
    """
)

┌────────────────────────────────────────┬─────────────────────┬───────┬───────────────┬─────────┐
│                   id                   │        date         │ year  │ property_type │  price  │
│                varchar                 │      timestamp      │ int64 │    varchar    │  int64  │
├────────────────────────────────────────┼─────────────────────┼───────┼───────────────┼─────────┤
│ {2131FCF5-B031-86E8-E063-4804A8C0372B} │ 2024-07-26 00:00:00 │  2024 │ Terraced      │  320000 │
│ {2131FCF5-B034-86E8-E063-4804A8C0372B} │ 2024-02-15 00:00:00 │  2024 │ Semi-Detached │  300000 │
│ {2131FCF5-B036-86E8-E063-4804A8C0372B} │ 2024-08-21 00:00:00 │  2024 │ Semi-Detached │  470000 │
│ {2131FCF5-B037-86E8-E063-4804A8C0372B} │ 2024-07-29 00:00:00 │  2024 │ Detached      │  527500 │
│ {2131FCF5-B038-86E8-E063-4804A8C0372B} │ 2024-07-17 00:00:00 │  2024 │ Terraced      │  351000 │
│ {2131FCF5-B03A-86E8-E063-4804A8C0372B} │ 2024-08-02 00:00:00 │  2024 │ Detached      │  395000 │
│ {2131FCF

In [35]:
annual_price_by_property_type = duckdb.sql(
    """
    SELECT
        year,
        property_type,
        MEDIAN(price) AS median_price
    FROM land_registry_data
    GROUP BY year, property_type
    ORDER BY year, property_type
    """
)

In [36]:
annual_price_by_property_type

┌───────┬───────────────┬──────────────┐
│ year  │ property_type │ median_price │
│ int64 │    varchar    │    double    │
├───────┼───────────────┼──────────────┤
│  2024 │ Detached      │     410000.0 │
│  2024 │ Flat          │     235000.0 │
│  2024 │ Semi-Detached │     262500.0 │
│  2024 │ Terraced      │     220000.0 │
│  2025 │ Detached      │     416100.5 │
│  2025 │ Flat          │     230000.0 │
│  2025 │ Semi-Detached │     270000.0 │
│  2025 │ Terraced      │     225000.0 │
│  2026 │ Detached      │     416000.0 │
│  2026 │ Flat          │     215000.0 │
│  2026 │ Semi-Detached │     270000.0 │
│  2026 │ Terraced      │     225000.0 │
└───────┴───────────────┴──────────────┘
  12 rows                    3 columns

## Visualise the results

In [37]:
annual_price_by_property_type.pl()

year,property_type,median_price
i64,str,f64
2024,"""Detached""",410000.0
2024,"""Flat""",235000.0
2024,"""Semi-Detached""",262500.0
2024,"""Terraced""",220000.0
2025,"""Detached""",416100.5
…,…,…
2025,"""Terraced""",225000.0
2026,"""Detached""",416000.0
2026,"""Flat""",215000.0


In [38]:
import plotly.express as px

In [39]:
px.line(
    annual_price_by_property_type.pl(),
    x="year",
    y="median_price",
    color="property_type",
    markers=True,
    title="Median Price by Year and Property Type",
    width=800,
    height=600,
)

In [40]:
from pathlib import Path
Path("../../data/price_paid_insights").mkdir(parents=True, exist_ok=True)

duckdb.sql(
    """
    COPY annual_price_by_property_type
    TO '../../data/price_paid_insights/annual_price_by_property_type.parquet'
    (FORMAT PARQUET)
    """
)

In [41]:
duckdb.sql(
    f"""
    SELECT
        year(column02) AS year,
        CASE column04
            WHEN 'D' THEN 'Detached'
            WHEN 'S' THEN 'Semi-Detached'
            WHEN 'T' THEN 'Terraced'
            WHEN 'F' THEN 'Flat'
            WHEN 'O' THEN 'Other'
            ELSE column04
        END AS property_type,
        MEDIAN(column01) AS median_price
    FROM read_csv('{DATA_PATH}/pp-*.csv', header = false)
    WHERE column04 != 'O'
    GROUP BY ALL
    ORDER BY year, property_type
    """
)

┌───────┬───────────────┬──────────────┐
│ year  │ property_type │ median_price │
│ int64 │    varchar    │    double    │
├───────┼───────────────┼──────────────┤
│  2024 │ Detached      │     410000.0 │
│  2024 │ Flat          │     235000.0 │
│  2024 │ Semi-Detached │     262500.0 │
│  2024 │ Terraced      │     220000.0 │
│  2025 │ Detached      │     416100.5 │
│  2025 │ Flat          │     230000.0 │
│  2025 │ Semi-Detached │     270000.0 │
│  2025 │ Terraced      │     225000.0 │
│  2026 │ Detached      │     416000.0 │
│  2026 │ Flat          │     215000.0 │
│  2026 │ Semi-Detached │     270000.0 │
│  2026 │ Terraced      │     225000.0 │
└───────┴───────────────┴──────────────┘
  12 rows                    3 columns

In [42]:
full_query = f"""
    SELECT
        year(column02) AS year,
        CASE column04
            WHEN 'D' THEN 'Detached'
            WHEN 'S' THEN 'Semi-Detached'
            WHEN 'T' THEN 'Terraced'
            WHEN 'F' THEN 'Flat'
            WHEN 'O' THEN 'Other'
            ELSE column04
        END AS property_type,
        MEDIAN(column01) AS median_price
    FROM read_csv('{DATA_PATH}/pp-*.csv', header = false)
    WHERE column04 != 'O'
    GROUP BY ALL
    ORDER BY year, property_type
"""

In [43]:
duckdb.sql(f"EXPLAIN {full_query}")

┌───────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [44]:
duckdb.sql(full_query)

┌───────┬───────────────┬──────────────┐
│ year  │ property_type │ median_price │
│ int64 │    varchar    │    double    │
├───────┼───────────────┼──────────────┤
│  2024 │ Detached      │     410000.0 │
│  2024 │ Flat          │     235000.0 │
│  2024 │ Semi-Detached │     262500.0 │
│  2024 │ Terraced      │     220000.0 │
│  2025 │ Detached      │     416100.5 │
│  2025 │ Flat          │     230000.0 │
│  2025 │ Semi-Detached │     270000.0 │
│  2025 │ Terraced      │     225000.0 │
│  2026 │ Detached      │     416000.0 │
│  2026 │ Flat          │     215000.0 │
│  2026 │ Semi-Detached │     270000.0 │
│  2026 │ Terraced      │     225000.0 │
└───────┴───────────────┴──────────────┘
  12 rows                    3 columns